Notebook adapted from Iván Pulido, Sukrit Singh, OpenFE docs, AlchemicalAnalysis (Mobley Lab)

# Functions

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import glob
import os
import matplotlib.pyplot as plt

## Energetic Analysis
- $\Delta\Delta G$ and uncertainty estimates using BAR
- work trajectories and distributions
- dU/d$\lambda$

### Estimating $\Delta\Delta G$ and Uncertainty using PyMBAR
- ```is_cycle_complete``` : check if given CycleUnit directory has completed results
- ```get_num_cycles``` : get minimum number of completed cycles across mutations to compare BAR estimates
- ```load_work_arrays``` : load work values stored in npy arrays of completed cycles
- ```plot_works``` : plot work trajectories and distributions
- ```compute_ddG_estimate``` : compute ddG and uncertainty using MBAR

In [2]:
def is_cycle_complete(cycle_dir):
    """Checks if given CycleUnit directory has completed results.
    Inputs:
        - cycle_dir (str) : path to CycleUnit directory
    Outputs:
        - (bool) : True if CycleUnit completed, False if not
    """
    npy_files = glob.glob(f"{cycle_dir}/**.npy", recursive=True)
    return len(npy_files) != 0 

In [3]:
def get_num_cycles(mutation_list, data_dir):
    """Get minimum number of completed cycles to compare BAR estimates across mutations.
    Inputs:
        - mutation_list (list) : list of mutations
        - data_dir (str) : path to data directory
    Outputs:
        - cycle_counts (dict) : counts of completed cycles for apo and holo legs
        - min_num_cycles (int) : minimal number of completed cycles
    """
    cycle_counts, min_num_cycles = {}, np.inf
    for mutation in mutation_list:
        apo_dir, holo_dir = f"{data_dir}/apo", f"{data_dir}/holo"
        apo_cycles = glob.glob(f"{apo_dir}/results_{mutation}/shared_CycleUnit**")
        holo_cycles = glob.glob(f"{holo_dir}/results_{mutation}/shared_CycleUnit**")
    
        apo_completed = sum([is_cycle_complete(cycle) for cycle in apo_cycles])
        holo_completed = sum([is_cycle_complete(cycle) for cycle in holo_cycles])

        min_num_cycles = min(min_num_cycles, min(apo_completed, holo_completed))
        cycle_counts[mutation] = {'apo': apo_completed, 'holo': holo_completed}
    return cycle_counts, min_num_cycles

In [4]:
def load_work_arrays(directory, direction):
    """ Load works stored in numpy arrays. """

    def subtract_offset(work):
        """ Subtract the initial work of the work trajectory. """
        work_offset = []
        for cycle in work:
            work_offset.append(np.array([value - cycle[0] for value in cycle[1:]]))
        work_offset = np.array(work_offset)
        return work_offset
    
    paths = []
    npy_files = glob.glob(f"{directory}/shared_CycleUnit**/{direction}_cycle_**.npy", recursive=True)

    work_arrays = []
    for path in npy_files:
        with open(path, 'rb') as rf:
            work_arrays.append(np.load(rf))

    if work_arrays == []:
        return np.nan, np.nan
    
    # compute this separately because the last value of the subsampled array is diff than the actual last sample
    work_arrays_combined = np.concatenate(work_arrays)
    work_arrays_accumulated = np.array([cycle[-1] - cycle[0] for cycle in work_arrays])
    work_arrays_combined = np.array([cycle for cycle in work_arrays])
    work_arrays_offset = subtract_offset(work_arrays_combined)

    return work_arrays_offset, work_arrays_accumulated

In [5]:
def plot_works(
        forward_work_offset, reverse_work_offset,
        dg, ddg,
        switching_time,
        title,
        output_dir
):
    """ Plot the work trajectory and distribution. """
    import matplotlib.pyplot as plt
    import seaborn as sns

    title_list = title_list = " ".join(title.split()).split(" ") 
    image_name = f"{title_list[0]}_{title_list[1]}"

    # plot work trajectories
    for i, cycle in enumerate(forward_work_offset):
        x = [(x + 1) * (switching_time / len(cycle)) for x in range(len(cycle))]
        if i == 0:
            plt.plot(x, cycle, color="#377eb8", label="forward")
        else:
            plt.plot(x, cycle, color="#377eb8")

    for i, cycle in enumerate(reverse_work_offset):
        x = [(x + 1) * (switching_time / len(cycle)) for x in range(len(cycle))]
        if i == 0:
            plt.plot(x, -cycle, color="#ff7f00", label="reverse")
        else:
            plt.plot(x, -cycle, color="#ff7f00")

    plt.xlabel("$t_{NEQ}$ [ns]")
    plt.ylabel("work [kT]")
    plt.title(title)
    plt.legend(loc='best')
    plt.savefig(f"{output_dir}/{image_name}_work_trajectory.png", dpi=500)
    plt.clf()

    # plot work distributions
    accumulated_forward = [cycle[-1] for cycle in forward_work_offset]
    accumulated_reverse = [-cycle[-1] for cycle in reverse_work_offset]
    min_work = int(min(accumulated_forward + accumulated_reverse)) - 4
    max_work = int(max(accumulated_forward + accumulated_reverse) + 5)
    bins = range(min_work, max_work)
    forward_ax = sns.histplot(
        accumulated_forward,
        color="#377eb8",
        label="forward",
        stat="probability",
        bins=bins,
        kde=True,
        kde_kws={"cut": 3},
        alpha=0.4,
        linewidth=0,
    )
    reverse_ax = sns.histplot(
        accumulated_reverse,
        color="#ff7f00",
        label="reverse",
        stat="probability",
        bins=bins,
        kde=True,
        kde_kws={"cut": 3},
        alpha=0.4,
        linewidth=0,
    )
    # Extract KDE data
    kde1_x, kde1_y = forward_ax.lines[0].get_xydata().T
    kde2_x, kde2_y = reverse_ax.lines[1].get_xydata().T

    # Interpolate the second KDE to match x-values of the first
    from scipy.interpolate import interp1d
    kde2_interp = interp1d(kde2_x, kde2_y, bounds_error=False, fill_value=0)
    kde2_y_interp = kde2_interp(kde1_x)


    plt.axvline(dg)
    plt.axvline(dg + ddg, linestyle='dashed')
    plt.axvline(dg - ddg, linestyle='dashed')
    plt.xticks(bins[::int(len(bins) / 8)])
    plt.xlabel(f"work [kT]")
    plt.ylabel("P(w)")
    plt.title(title)
    plt.legend(loc='best')
    plt.savefig(f"{output_dir}/{image_name}_work_distribution.png", dpi=500)
    plt.clf()

In [6]:
def compute_ddG_estimate(mutation_list, data_dir, num_cycles=None, plot_dir=None, switching_time=1.5):
    """Compute ddG and uncertainty using MBAR.
    Inputs:
        - mutation_list (list) : list of mutations
        - data_dir (str) : path to data directory
        - num_cycles (int) : optional minimal number of cycles to compare BAR estimates across mutations
        - plot_dir (str) : optional path to directory for plotted work trajectories and distributions; default is current directory
        - switching_time (float) : optional NEQ time; default is 1.5 ns
    Outputs:
        - fe_estimates_df (Dataframe) : overall, apo, and holo free energy and uncertainty estimates
    """
    from pymbar import bar
    
    fe_estimates = {}
    if plot_dir:
        os.makedirs(plot_dir, exist_ok=True) # check if plot_dir exists
    else: 
        plot_dir = os.getcwd() # use current dir for plotting if none specified
        
    for mutation in mutations:
        print(f"Processing {mutation}")    
        apo_dir, holo_dir = f"{data_dir}/apo/results_{mutation}", f"{data_dir}/holo/results_{mutation}"
    
        forward_apo_offset, forward_apo_accumulated = load_work_arrays(apo_dir, "forward")
        reverse_apo_offset, reverse_apo_accumulated = load_work_arrays(apo_dir, "reverse")
        forward_holo_offset, forward_holo_accumulated = load_work_arrays(holo_dir, "forward")
        reverse_holo_offset, reverse_holo_accumulated = load_work_arrays(holo_dir, "reverse")

        apo_results = bar.BAR(forward_apo_accumulated[:num_cycles], reverse_apo_accumulated[:num_cycles])
        holo_results = bar.BAR(forward_holo_accumulated[:num_cycles], reverse_holo_accumulated[:num_cycles])
    
        overall_results = ( (holo_results[0] - apo_results[0]),
                           np.sqrt(holo_results[1]**2 + apo_results[1]**2) )

        fe_estimates[mutation] = {"fe_estimate_kcalmol": overall_results[0], "fe_uncertainty_kcalmol": overall_results[1],
                                       "apo_estimate_kcalmol": apo_results[0], "apo_uncertainty_kcalmol": apo_results[1],
                                       "holo_estimate_kcalmol": holo_results[0], "holo_uncertainty_kcalmol": holo_results[1]}

        plot_works(
            forward_apo_offset[:num_cycles], reverse_apo_offset[:num_cycles],
            apo_results[0], apo_results[1],
            switching_time=switching_time,
            title=f"{mutation} Apo Leg ({num_cycles} Cycles)",
            output_dir=plot_dir)
    
        plot_works(
            forward_holo_offset[:num_cycles], reverse_holo_offset[:num_cycles],
            holo_results[0], holo_results[1],
            switching_time=switching_time,
            title=f"{mutation} Holo Leg ({num_cycles} Cycles)",
            output_dir=plot_dir)
    
    fe_estimates_df = pd.DataFrame.from_dict(fe_estimates, orient='index')
    return fe_estimates_df

### Plotting Convergence (dU/d$\lambda$)

## Structural Analysis
- protein RMSD
- ligand RMSD
- per-residue protein RMSF

### Constructing Trajectories
- ```get_HTF_data```: returns HTF topology, initial and final indices in HTF
- ```make_trajectory```: given prefix ('old' or 'new') and CycleUnit directory, load and concatentate all npy files corresponding to prefix into one trajectory, return concatenated trajectory

In [7]:
import mdtraj as md

In [8]:
def get_HTF_data(setup_dir):
    """
    Initialize trajectory data dictionary with HTF initial and final state topologies.
    Input: 
        - setup_dir (str) : path to SetupUnit directory 
    Output: 
        - htf_dict (dict) : topologies and reference trajectories for initial and final states (specific to mutation)
    """
    import pickle
    htf_path = f'{setup_dir}/hybrid_topology_factory.pickle'
    with open(htf_path, 'rb') as f:
        htf = pickle.load(f)

    # Subset HTF topology by initial and final states
    initial_top = htf.hybrid_topology.subset(htf.initial_atom_indices)
    final_top = htf.hybrid_topology.subset(htf.final_atom_indices)

    initial_xyz = htf.old_positions(htf.hybrid_positions)
    final_xyz = htf.new_positions(htf.hybrid_positions)
    
    # Initialize trajectory data dictionary with topologies
    htf_dict = {'old_topology': initial_top,
                'new_topology': final_top,
                'old_reference': md.Trajectory(initial_xyz, initial_top),
                'new_reference': md.Trajectory(final_xyz, final_top)
                }
    return htf_dict

In [9]:
def filter_CycleUnit_dirs(results_dir):
    """
    Find all CycleUnits with results.
    Inputs:
        - results_dir (str) : path to files for mutation
    Outputs:
        - cycle_data (list) : list of valid CycleUnit directory paths with results in npy arrays
    """
    def filter_dir(dirpath):
        import os
        dir_files = [entry for entry in os.listdir(dirpath) if os.path.isfile(os.path.join(dirpath, entry))]
        npy_files = [file for file in dir_files if ".npy" in file]
        if len(npy_files) == 10: # 10 npy files for valid results 
            return dirpath
            
    result_cycles = glob.glob(f'{results_dir}/shared_CycleUnit**')
    filtered_cycles = [filter_dir(cycle) for cycle in result_cycles]
    filtered_cycles = list(filter(None, filtered_cycles))
    return filtered_cycles

In [10]:
def get_NEQ_traj(cycle_dir, htf_dict):
    """
    Append trajectory data dictionary with all NEQ trajectories.
    Inputs: 
        - cycle_dir (str) : path to CycleUnit directory
        - htf_dict (dict) : dictionary with topologies for initial and final states
    Output: 
        - traj_dict (dict) : trajectories added (specific to each CycleUnit for mutation)
    """
    traj_dict = {}
    for state in states:
        for direction in directions:
            for phase in phases:
                # Load npy arrays with coordinates and frames with the topologies 
                file_path = glob.glob(f'{cycle_dir}/{direction}_{phase}_{state}_cycle_*.npy')[0]
                xyz = np.load(file_path)
                traj_dict[f'{direction}_{phase}_{state}'] = md.Trajectory(xyz, htf_dict[f'{state}_topology'])

        keys = [key for key in list(traj_dict.keys()) if state in key]
        trajs_to_join = [traj_dict[key] for key in keys]
        combined_traj = md.join(trajs_to_join)
        traj_dict[f'{state}_trajectory'] = combined_traj
    return traj_dict

In [11]:
def compute_traj_rmsd(traj_dict, htf_dict, state):
    """Compute RMSD"""
    return md.rmsd(traj_dict[f'{state}_trajectory'], htf_dict[f'{state}_reference'])

# Usage

## Energetic Analysis

### User-Defined Parameters

In [12]:
# Path to Abl-ATP NEQ directory
data_path = "/data1/choderaj/pathilm/FEC/kinase_ATP/abl"

# Mutations for Abl kinase
mutations = ["THR-315-ILE",
            "GLY-250-GLU",
            "TYR-253-PHE",
            "PHE-317-LEU",
            "ASN-368-SER"]

# NEQ switching time
time = 1.5 # 1.5 ns, 375000 timesteps

# Directory for plots
test_plot_dir = "plots"
test_plot_dir = f"{data_path}/presentation_images/abl_atp_analysis"

### $\Delta\Delta G$ Estimates

In [13]:
counts, cycles = get_num_cycles(mutations, data_path)
counts, cycles

({'THR-315-ILE': {'apo': 148, 'holo': 155},
  'GLY-250-GLU': {'apo': 167, 'holo': 183},
  'TYR-253-PHE': {'apo': 139, 'holo': 149},
  'PHE-317-LEU': {'apo': 131, 'holo': 148},
  'ASN-368-SER': {'apo': 90, 'holo': 90}},
 90)

In [14]:
fe_estimates = compute_ddG_estimate(mutations, data_path, num_cycles=cycles, plot_dir=test_plot_dir)
fe_estimates

Processing THR-315-ILE
Processing GLY-250-GLU
Processing TYR-253-PHE
Processing PHE-317-LEU
Processing ASN-368-SER


,fe_estimate_kcalmol,fe_uncertainty_kcalmol,apo_estimate_kcalmol,apo_uncertainty_kcalmol,holo_estimate_kcalmol,holo_uncertainty_kcalmol
THR-315-ILE,-3.952322,0.407175,-27.246595,0.314182,-31.198917,0.259000
GLY-250-GLU,9.054241,0.562004,-279.243752,0.337854,-270.189511,0.449114
TYR-253-PHE,-0.413522,0.100911,-2.228002,0.073424,-2.641524,0.069224
PHE-317-LEU,2.070063,0.219609,9.899016,0.140229,11.969079,0.169009
ASN-368-SER,1.328318,1.002852,-22.447122,0.896925,-21.118804,0.448594


<Figure size 640x480 with 0 Axes>

## Structural Analysis

### User-Defined Parameters

In [15]:
# Use whatever labels for initial and final states, directions of trajectory, and phases
states = ['old', 'new'] # should be ['initial', 'final']
directions = ['forward', 'reverse']
phases = ['eq', 'neq']

# Sample results directory
results = '/data1/choderaj/pathilm/FEC/kinase_ATP/abl/holo/results_TYR-253-PHE'
cycle_unit = f'{results}/shared_CycleUnit-fef55b9ac866466d83a6749c4ccacd6f_attempt_0' 
setup_unit = f'{results}/shared_SetupUnit-fb4035e22f834972a62a13dac35be899_attempt_0'

### Trajectories

In [16]:
# Store initial and final topologies using HTF data
htf_data = get_HTF_data(setup_unit)
htf_data

/usersoftware/choderaj/pathilm/miniforge3/envs/pale-env/lib/python3.12/site-packages/mdtraj/core/topology.py:84: UserWarning: atom_indices are not monotonically increasing
  warnings.warn("atom_indices are not monotonically increasing")


{'old_topology': <mdtraj.Topology with 6 chains, 14834 residues, 47651 atoms, 33124 bonds at 0x7efea19fe480>,
 'new_topology': <mdtraj.Topology with 6 chains, 14834 residues, 47650 atoms, 33123 bonds at 0x7efeab36b410>,
 'old_reference': <mdtraj.Trajectory with 1 frames, 47651 atoms, 14834 residues, without unitcells at 0x7efe9f6f5820>,
 'new_reference': <mdtraj.Trajectory with 1 frames, 47650 atoms, 14834 residues, without unitcells at 0x7efeab5f6ff0>}

In [17]:
# Filter CycleUnit directories for only directories with completed cycles
cycles = filter_CycleUnit_dirs(results)

In [18]:
# Store all trajectories for all completed CycleUnits
traj_dicts = {}
for cycle in cycles:
    traj_dicts[cycle] = get_NEQ_traj(cycle, htf_data)

In [19]:
cycles

['/data1/choderaj/pathilm/FEC/kinase_ATP/abl/holo/results_TYR-253-PHE/shared_CycleUnit-7ea57630ab444613b52383e813b238df_attempt_0',
 '/data1/choderaj/pathilm/FEC/kinase_ATP/abl/holo/results_TYR-253-PHE/shared_CycleUnit-4d5e3925dd9f48529d73a80d886e4eb2_attempt_0',
 '/data1/choderaj/pathilm/FEC/kinase_ATP/abl/holo/results_TYR-253-PHE/shared_CycleUnit-c3b82cee6b9c4a86a73612fcda3979d9_attempt_0',
 '/data1/choderaj/pathilm/FEC/kinase_ATP/abl/holo/results_TYR-253-PHE/shared_CycleUnit-9ec9246dda90408fa13448ef479168d1_attempt_0',
 '/data1/choderaj/pathilm/FEC/kinase_ATP/abl/holo/results_TYR-253-PHE/shared_CycleUnit-ccfd4e5f3dca4c969d32676b8b3924bb_attempt_0',
 '/data1/choderaj/pathilm/FEC/kinase_ATP/abl/holo/results_TYR-253-PHE/shared_CycleUnit-61a420f2916540a2ac84424103197b20_attempt_0',
 '/data1/choderaj/pathilm/FEC/kinase_ATP/abl/holo/results_TYR-253-PHE/shared_CycleUnit-e58142b4a7c74455af2799c225dbf99c_attempt_0',
 '/data1/choderaj/pathilm/FEC/kinase_ATP/abl/holo/results_TYR-253-PHE/shared

In [20]:
len(cycles)

149

In [21]:
traj_dicts

{'/data1/choderaj/pathilm/FEC/kinase_ATP/abl/holo/results_TYR-253-PHE/shared_CycleUnit-7ea57630ab444613b52383e813b238df_attempt_0': {'forward_eq_old': <mdtraj.Trajectory with 3 frames, 47651 atoms, 14834 residues, without unitcells at 0x7efeab01ca10>,
  'forward_neq_old': <mdtraj.Trajectory with 2 frames, 47651 atoms, 14834 residues, without unitcells at 0x7efe8fc341d0>,
  'reverse_eq_old': <mdtraj.Trajectory with 2 frames, 47651 atoms, 14834 residues, without unitcells at 0x7efe9c137e30>,
  'reverse_neq_old': <mdtraj.Trajectory with 1 frames, 47651 atoms, 14834 residues, without unitcells at 0x7efe9c137ec0>,
  'old_trajectory': <mdtraj.Trajectory with 8 frames, 47651 atoms, 14834 residues, without unitcells at 0x7efe9c137e90>,
  'forward_eq_new': <mdtraj.Trajectory with 3 frames, 47650 atoms, 14834 residues, without unitcells at 0x7efe9f95ea20>,
  'forward_neq_new': <mdtraj.Trajectory with 2 frames, 47650 atoms, 14834 residues, without unitcells at 0x7efe9f95ca40>,
  'reverse_eq_new':